# Data Quality Assessment

## Business Objective

The objective of this notebook is to assess the quality and reliability of the raw e-commerce datasets before any cleaning, integration, or analysis is performed.

This assessment evaluates the following six tables:

- `users`
- `products`
- `orders`
- `order_items`
- `reviews`
- `events`

The purpose is to identify data quality issues such as missing values, duplicate records, invalid values, inconsistent formatting, incorrect datatypes, and primary key violations. No cleaning operations are performed in this notebook. All identified issues will be addressed later in the Data Cleaning phase.

## Methodology

Each dataset is assessed using a consistent framework:

1. Dataset structure  
2. Column overview  
3. Datatype inspection  
4. Missing value assessment  
5. Duplicate row assessment  
6. Primary key uniqueness check  
7. Business-rule validation  
8. Summary of key findings  

This structure ensures that each table is evaluated consistently and that cleaning decisions are based on documented evidence rather than assumptions.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Set Project Directory

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

PROJECT_ROOT

PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis')

## Verify Available Raw Datasets

In [3]:
list(DATA_DIR.iterdir())

[PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/reviews_raw.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/products_raw.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/users_raw.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/events_raw.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/orders_raw.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/raw/order_items_raw.csv')]

## Load Raw Datasets

In [4]:
users = pd.read_csv(DATA_DIR / "users_raw.csv")
products = pd.read_csv(DATA_DIR / "products_raw.csv")
orders = pd.read_csv(DATA_DIR / "orders_raw.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_raw.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_raw.csv")
events = pd.read_csv(DATA_DIR / "events_raw.csv")

## Helper Functions

To avoid repeating the same code for every table, reusable helper functions are created below. These functions summarize structure, missing values, duplicate records, and primary key uniqueness for each dataset.

In [5]:
def dataset_overview(df, table_name):
    print(f"===== {table_name.upper()} DATASET OVERVIEW =====")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")
    print("\nColumn Names:")
    print(list(df.columns))
    print("\nData Types:")
    print(df.dtypes)


def missing_values_report(df):
    missing_count = df.isna().sum()
    missing_percent = (df.isna().mean() * 100).round(2)

    report = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    })

    return report[report["missing_count"] > 0].sort_values(
        by="missing_percent",
        ascending=False
    )


def duplicate_rows_report(df):
    duplicate_count = df.duplicated().sum()
    duplicate_percent = round(df.duplicated().mean() * 100, 2)

    return pd.DataFrame({
        "duplicate_rows": [duplicate_count],
        "duplicate_percent": [duplicate_percent]
    })


def primary_key_report(df, key_column):
    total_rows = len(df)
    unique_keys = df[key_column].nunique(dropna=False)
    duplicated_keys = df[key_column].duplicated().sum()
    missing_keys = df[key_column].isna().sum()

    return pd.DataFrame({
        "key_column": [key_column],
        "total_rows": [total_rows],
        "unique_keys": [unique_keys],
        "duplicated_keys": [duplicated_keys],
        "missing_keys": [missing_keys]
    })

# 1. Users Data Quality Assessment

## Business Context

The Users table represents customer profile information. Each row should represent one unique customer. This table is important because it connects customer behavior, orders, reviews, and events through `user_id`.

## Expected Data Quality Rules

- `user_id` should uniquely identify each customer.
- `user_id` should not be missing.
- Exact duplicate records should not exist.
- `signup_date` should represent a valid date.
- Demographic fields such as `gender` and `city` should be checked for completeness and consistency.

In [6]:
dataset_overview(users, "users")

===== USERS DATASET OVERVIEW =====
Rows: 10,200
Columns: 6

Column Names:
['user_id', 'name', 'email', 'gender', 'city', 'signup_date']

Data Types:
user_id        object
name           object
email          object
gender         object
city           object
signup_date    object
dtype: object


## Users Missing Values

In [7]:
missing_values_report(users)

,missing_count,missing_percent
email,408,4.00
gender,408,4.00
city,365,3.58


## Users Duplicate Rows

In [8]:
duplicate_rows_report(users)

,duplicate_rows,duplicate_percent
0,127,1.25


## Users Primary Key Check

In [9]:
primary_key_report(users, "user_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,user_id,10200,10000,200,0


## Users Invalid Date Check

In [10]:
invalid_signup_dates = users[
    pd.to_datetime(users["signup_date"], errors="coerce").isna()
]

invalid_signup_dates.shape[0]

102

## Users Interpretation

The Users table should contain one record per customer. The assessment checks whether the customer identifier is unique, whether profile information is complete, whether duplicate rows exist, and whether the `signup_date` field can be converted into a valid datetime format.

Any duplicate customer identifiers, missing profile fields, or invalid signup dates should be addressed during the Data Cleaning phase.

# 2. Products Data Quality Assessment

## Business Context

The Products table contains the product catalog. This table is central to sales, category, revenue, and product performance analysis.

## Expected Data Quality Rules

- `product_id` should uniquely identify each product.
- `product_id` should not be missing.
- Exact duplicate product records should not exist.
- `price` should be positive.
- `rating` should be within a valid range, typically 1 to 5.
- Product categories should be consistent.

In [11]:
dataset_overview(products, "products")

===== PRODUCTS DATASET OVERVIEW =====
Rows: 2,040
Columns: 6

Column Names:
['product_id', 'product_name', 'category', 'brand', 'price', 'rating']

Data Types:
product_id       object
product_name     object
category         object
brand            object
price           float64
rating          float64
dtype: object


## Products Missing Values

In [12]:
missing_values_report(products)

,missing_count,missing_percent
product_name,61,2.99
rating,60,2.94
category,53,2.60


## Products Duplicate Rows

In [13]:
duplicate_rows_report(products)

,duplicate_rows,duplicate_percent
0,27,1.32


## Products Primary Key Check

In [14]:
primary_key_report(products, "product_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,product_id,2040,2000,40,0


## Products Price Validity Check

In [15]:
invalid_prices = products[products["price"] <= 0]
invalid_prices.shape[0]

20

## Products Rating Validity Check

In [16]:
invalid_product_ratings = products[
    (products["rating"] < 1) | (products["rating"] > 5)
]

invalid_product_ratings.shape[0]

20

## Products Category Consistency Check

In [17]:
products["category"].value_counts(dropna=False)

category
Clothing             197
Pet Supplies         193
Beauty               193
Toys                 185
Electronics          183
Home & Kitchen       182
Automotive           181
Books                174
Sports               168
Groceries            168
NaN                   53
  toys                24
  electronics         18
  home & kitchen      17
  books               17
  pet supplies        16
  clothing            16
  automotive          14
  sports              14
  groceries           12
  nan                  8
  beauty               7
Name: count, dtype: int64

## Products Interpretation

The Products table is expected to provide a reliable product catalog for revenue and category-level analysis. Key risks include duplicate product identifiers, missing product information, invalid prices, ratings outside the accepted range, and inconsistent category labels.

These issues may affect product-level reporting and business intelligence dashboards if not corrected during cleaning.

# 3. Orders Data Quality Assessment

## Business Context

The Orders table contains order-level transactions. It is essential for analyzing revenue, order volume, customer purchasing behavior, and business growth over time.

## Expected Data Quality Rules

- `order_id` should uniquely identify each order.
- `order_id` should not be missing.
- `user_id` should link each order to a valid customer.
- `order_date` should be a valid datetime value.
- `order_status` should contain valid business statuses.
- `total_amount` should not be negative.

In [18]:
dataset_overview(orders, "orders")

===== ORDERS DATASET OVERVIEW =====
Rows: 20,400
Columns: 5

Column Names:
['order_id', 'user_id', 'order_date', 'order_status', 'total_amount']

Data Types:
order_id         object
user_id          object
order_date       object
order_status     object
total_amount    float64
dtype: object


## Orders Missing Values

In [19]:
missing_values_report(orders)

,missing_count,missing_percent
total_amount,612,3.00
order_status,579,2.84


## Orders Duplicate Rows

In [20]:
duplicate_rows_report(orders)

,duplicate_rows,duplicate_percent
0,310,1.52


## Orders Primary Key Check

In [21]:
primary_key_report(orders, "order_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,order_id,20400,20000,400,0


## Orders Invalid Date Check

In [22]:
invalid_order_dates = orders[
    pd.to_datetime(orders["order_date"], errors="coerce").isna()
]

invalid_order_dates.shape[0]

204

## Orders Total Amount Validity Check

In [23]:
invalid_order_amounts = orders[orders["total_amount"] < 0]
invalid_order_amounts.shape[0]

198

## Orders Status Consistency Check

In [24]:
orders["order_status"].value_counts(dropna=False)

order_status
shipped          3828
returned         3779
completed        3753
cancelled        3626
processing       3611
NaN               579
  shipped         260
  cancelled       239
  processing      238
  completed       228
  returned        226
  nan              33
Name: count, dtype: int64

## Orders Interpretation

The Orders table is one of the most important transactional tables in the database. Data quality issues in this table can directly affect revenue reporting, sales trends, order status analysis, and customer purchasing insights.

Invalid dates, negative order amounts, missing statuses, duplicated order identifiers, or inconsistent order status labels should be investigated and cleaned before analysis.

# 4. Order Items Data Quality Assessment

## Business Context

The Order Items table contains item-level purchase details. It connects orders to products and enables detailed analysis of product sales, quantities, basket composition, and revenue contribution.

## Expected Data Quality Rules

- `order_item_id` should uniquely identify each order item.
- `order_id` should link each item to a valid order.
- `product_id` should link each item to a valid product.
- `quantity` should be greater than zero.
- `item_price` should be positive.
- Exact duplicate rows should not exist.

In [25]:
dataset_overview(order_items, "order_items")

===== ORDER_ITEMS DATASET OVERVIEW =====
Rows: 44,395
Columns: 7

Column Names:
['order_item_id', 'order_id', 'product_id', 'user_id', 'quantity', 'item_price', 'item_total']

Data Types:
order_item_id     object
order_id          object
product_id        object
user_id           object
quantity         float64
item_price       float64
item_total       float64
dtype: object


## Order Items Missing Values

In [26]:
missing_values_report(order_items)

,missing_count,missing_percent
item_price,888,2.00
quantity,876,1.97


## Order Items Duplicate Rows

In [27]:
duplicate_rows_report(order_items)

,duplicate_rows,duplicate_percent
0,768,1.73


## Order Items Primary Key Check

In [28]:
primary_key_report(order_items, "order_item_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,order_item_id,44395,43525,870,0


## Quantity Validity Check

In [29]:
invalid_quantities = order_items[order_items["quantity"] <= 0]
invalid_quantities.shape[0]

444

## Item Price Validity Check

In [30]:
invalid_item_prices = order_items[order_items["item_price"] <= 0]
invalid_item_prices.shape[0]

437

## Order Items Interpretation

The Order Items table determines product-level sales and revenue calculations. Invalid quantities or item prices may distort revenue and product performance metrics. Referential integrity with Orders and Products should also be validated during the Data Integration phase.

# 5. Reviews Data Quality Assessment

## Business Context

The Reviews table contains customer product feedback. It supports customer satisfaction analysis, product quality evaluation, and rating-based insights.

## Expected Data Quality Rules

- `review_id` should uniquely identify each review.
- `user_id` should link each review to a valid customer.
- `product_id` should link each review to a valid product.
- `rating` should be within a valid range, typically 1 to 5.
- `review_date` should be a valid date.
- Review text may be missing, but missing values should be documented.

In [31]:
dataset_overview(reviews, "reviews")

===== REVIEWS DATASET OVERVIEW =====
Rows: 15,300
Columns: 7

Column Names:
['review_id', 'order_id', 'product_id', 'user_id', 'rating', 'review_text', 'review_date']

Data Types:
review_id       object
order_id        object
product_id      object
user_id         object
rating         float64
review_text     object
review_date     object
dtype: object


## Reviews Missing Values

In [32]:
missing_values_report(reviews)

,missing_count,missing_percent
review_text,612,4.00
rating,607,3.97


## Reviews Duplicate Rows

In [33]:
duplicate_rows_report(reviews)

,duplicate_rows,duplicate_percent
0,253,1.65


## Reviews Primary Key Check

In [34]:
primary_key_report(reviews, "review_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,review_id,15300,15000,300,0


## Review Rating Validity Check

In [35]:
invalid_review_ratings = reviews[
    (reviews["rating"] < 1) | (reviews["rating"] > 5)
]

invalid_review_ratings.shape[0]

153

## Review Date Validity Check

In [36]:
invalid_review_dates = reviews[
    pd.to_datetime(reviews["review_date"], errors="coerce").isna()
]

invalid_review_dates.shape[0]

153

## Reviews Interpretation

The Reviews table is important for understanding customer satisfaction and product perception. Invalid ratings, invalid review dates, duplicate review identifiers, or missing review text should be documented and handled appropriately during cleaning.

# 6. Events Data Quality Assessment

## Business Context

The Events table captures user behavior such as product views, cart additions, wishlist actions, and purchases. It supports funnel analysis, conversion tracking, and behavioral analytics.

## Expected Data Quality Rules

- `event_id` should uniquely identify each event.
- `user_id` should link each event to a valid user.
- `product_id` should link each event to a valid product when applicable.
- `event_type` should contain valid event categories.
- `event_timestamp` should be a valid datetime value.
- Exact duplicate event records should not exist.

In [37]:
dataset_overview(events, "events")

===== EVENTS DATASET OVERVIEW =====
Rows: 81,600
Columns: 5

Column Names:
['event_id', 'user_id', 'product_id', 'event_type', 'event_timestamp']

Data Types:
event_id           object
user_id            object
product_id         object
event_type         object
event_timestamp    object
dtype: object


## Events Missing Values

In [38]:
missing_values_report(events)

,missing_count,missing_percent
product_id,2448,3.00
event_type,2292,2.81


## Events Duplicate Rows

In [39]:
duplicate_rows_report(events)

,duplicate_rows,duplicate_percent
0,1229,1.51


## Events Primary Key Check

In [40]:
primary_key_report(events, "event_id")

,key_column,total_rows,unique_keys,duplicated_keys,missing_keys
0,event_id,81600,80000,1600,0


## Event Type Consistency Check

In [41]:
events["event_type"].value_counts(dropna=False)

event_type
view           52138
cart           11190
wishlist        7389
purchase        3695
  view          3304
NaN             2292
  cart           717
  wishlist       480
  purchase       239
  nan            156
Name: count, dtype: int64

## Event Timestamp Validity Check

In [42]:
invalid_event_timestamps = events[
    pd.to_datetime(events["event_timestamp"], errors="coerce").isna()
]

invalid_event_timestamps.shape[0]

816

## Events Interpretation

The Events table is critical for behavioral and funnel analysis. Invalid timestamps, missing event types, duplicate events, or inconsistent event labels may affect conversion rate calculations and customer journey analysis.

# Overall Data Quality Report

The data quality assessment identified issues across multiple tables, including potential missing values, duplicate records, invalid dates, invalid numeric values, inconsistent categorical formatting, and primary key violations.

These findings confirm that the raw datasets require a structured cleaning phase before they can be safely used for analysis, reporting, dashboard development, or business decision-making.

## Data Quality Summary Table

In [43]:
quality_summary = pd.DataFrame({
    "table": [
        "users",
        "products",
        "orders",
        "order_items",
        "reviews",
        "events"
    ],
    "rows": [
        len(users),
        len(products),
        len(orders),
        len(order_items),
        len(reviews),
        len(events)
    ],
    "columns": [
        users.shape[1],
        products.shape[1],
        orders.shape[1],
        order_items.shape[1],
        reviews.shape[1],
        events.shape[1]
    ],
    "duplicate_rows": [
        users.duplicated().sum(),
        products.duplicated().sum(),
        orders.duplicated().sum(),
        order_items.duplicated().sum(),
        reviews.duplicated().sum(),
        events.duplicated().sum()
    ],
    "missing_values_total": [
        users.isna().sum().sum(),
        products.isna().sum().sum(),
        orders.isna().sum().sum(),
        order_items.isna().sum().sum(),
        reviews.isna().sum().sum(),
        events.isna().sum().sum()
    ]
})

quality_summary

,table,rows,columns,duplicate_rows,missing_values_total
0,users,10200,6,127,1181
1,products,2040,6,27,174
2,orders,20400,5,310,1191
3,order_items,44395,7,768,1764
4,reviews,15300,7,253,1219
5,events,81600,5,1229,4740


# Executive Summary

## Objective

The objective of this notebook was to assess the quality of the raw e-commerce database before cleaning, integration, and analysis. Six datasets were reviewed: Users, Products, Orders, Order Items, Reviews, and Events.

## Key Assessment Areas

The assessment focused on:

- Dataset structure
- Missing values
- Duplicate records
- Primary key uniqueness
- Invalid dates
- Invalid numeric values
- Categorical consistency
- Business-rule violations

## Main Findings

The assessment confirmed that the raw datasets contain several data quality issues that must be addressed before analysis. These include missing values, duplicate rows, duplicate identifiers, invalid date values, invalid numeric values, and inconsistent categorical formatting.

## Recommendation

Before performing exploratory analysis, SQL analysis, or dashboard development, the identified issues should be handled in a dedicated Data Cleaning notebook. Cleaning should be performed systematically, with every transformation documented and validated.

## Next Steps

The next stage of the project is the Data Cleaning phase, where each table will be cleaned according to the issues identified in this assessment. After cleaning, the datasets will be validated and integrated to support downstream business analysis.

## Project Methodology

This project follows a structured analytical workflow:

**Raw Data**  
↓  
**Data Quality Assessment**  
↓  
**Data Cleaning**  
↓  
**Validation**  
↓  
**Data Integration**  
↓  
**Exploratory Data Analysis**  
↓  
**SQL Business Analysis**  
↓  
**Dashboard Development**  
↓  
**Executive Reporting**

This workflow ensures that all insights are built on reliable, transparent, and well-documented data.